### 전처리 함수화

- 기본으로 주어지는 것들 -> 컬럼 1줄 전처리
- 코드 작성할 때 고려해야 하는 것들
  - 첫번째 컬럼 조사했을 때 시군구별(1)이면 시군구 컬럼 2개고, 시군구별이면 컬럼 1개.. 그에 따라 다르게 움직여야 함
  - 작성 완료되면 일단 잘되는지 확인하기

In [1073]:
# 임포트 함수들 
import pandas as pd
import os
import numpy as np

In [1074]:
def remove_only_sido_rows(df: pd.DataFrame, region_col: str = "SGG_NAME") -> pd.DataFrame:
    """
    시도 단독 행을 제거하고, '충청북도 충북' 등의 잘못된 지역명도 함께 제거

    Parameters:
        df (pd.DataFrame): 원본 데이터프레임
        region_col (str): 시군구 전체 이름 컬럼명

    Returns:
        pd.DataFrame: 시도 단독 및 중복 명칭 제거된 DataFrame
    """
    # 추가적으로 충청북도 충북, 충청남도 충남 등 시도명이 두 번 반복된 잘못된 명칭 제거
    invalid_names = [
        "서울특별시 서울",
        "인천광역시 인천",
        "경기도 경기"
    ]
    df = df[~df[region_col].isin(invalid_names)]
    df = df.sort_values(by=region_col).reset_index(drop=True)

    return df


In [1075]:
# 특례시 하위 행정구역 제거
def remove_target_regions(df: pd.DataFrame, region_col: str = "SGG_NAME") -> pd.DataFrame:
    """
    시군구 통합 대상에 포함된 하위 행정구역을 제거하고,
    세종특별자치시 단독 행은 유지하며, 나머지 시도 단독 행은 제거
    이후 시군구 기준 정렬

    Parameters:
        df (pd.DataFrame): 처리 대상 데이터프레임
        region_col (str): 행정구역 전체 컬럼명

    Returns:
        pd.DataFrame: 필터링 및 정렬된 결과 DataFrame
    """

    # 제거할 행정구역 리스트
    regions_to_remove = [
    # 수원시
    "경기도 장안구", "경기도 권선구", "경기도 팔달구", "경기도 영통구",
    "경기도 수원시 장안구", "경기도 수원시 권선구", "경기도 수원시 팔달구", "경기도 수원시 영통구",
    "경기도 수원시장안구", "경기도 수원시권선구", "경기도 수원시팔달구", "경기도 수원시영통구",

    # 고양시
    "경기도 덕양구", "경기도 일산동구", "경기도 일산서구",
    "경기도 고양시 덕양구", "경기도 고양시 일산동구", "경기도 고양시 일산서구",
    "경기도 고양시덕양구", "경기도 고양시일산동구", "경기도 고양시일산서구",
    "경기도 고양시 덕양구", "경기도 고양시 일산 동구", "경기도 고양시 일산 서구",

    # 용인시
    "경기도 처인구", "경기도 기흥구", "경기도 수지구",
    "경기도 용인시 처인구", "경기도 용인시 기흥구", "경기도 용인시 수지구",
    "경기도 용인시처인구", "경기도 용인시기흥구", "경기도 용인시수지구",

    # 성남시
    "경기도 수정구", "경기도 중원구", "경기도 분당구",
    "경기도 성남시 수정구", "경기도 성남시 중원구", "경기도 성남시 분당구",
    "경기도 성남시수정구", "경기도 성남시중원구", "경기도 성남시분당구",

    # 안산시
    "경기도 상록구", "경기도 단원구",
    "경기도 안산시 상록구", "경기도 안산시 단원구",
    "경기도 안산시상록구", "경기도 안산시단원구",

    # 안양시
    "경기도 만안구", "경기도 동안구",
    "경기도 안양시 만안구", "경기도 안양시 동안구",
    "경기도 안양시만안구", "경기도 안양시동안구",
]

    # 특정 행정구역 제거
    df = df[~df[region_col].isin(regions_to_remove)]

    # 정렬 및 인덱스 초기화
    df = df.sort_values(by=region_col).reset_index(drop=True)
    return df


In [1076]:
# 시도 + 시군구 조합으로 'SGG_NAME' 컬럼 생성
def create_full_region_column(df: pd.DataFrame, column: str = "행정구역별(시군구)") -> pd.DataFrame:
    """
    시도 + 시군구 조합으로 'SGG_NAME' 컬럼 생성

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        column (str): 시도/시군구가 포함된 컬럼명 (기본값: '행정구역별(시군구)')

    Returns:
        pd.DataFrame: 'SGG_NAME' 컬럼이 추가된 DataFrame
    """
    cities = [
        "서울특별시", "인천광역시", "경기도"
    ]

    current_city = None
    new_names = []

    for region in df[column]:
        if region in cities:
            current_city = region
            new_names.append(region)
        else:
            new_names.append(f"{current_city} {region}")

    df["SGG_NAME"] = new_names
    # 기존 컬럼 제거
    df.drop(columns=[column], inplace=True)
    
    # 컬럼 순서 조정
    cols = ["SGG_NAME"] + [col for col in df.columns if col != "SGG_NAME"]
    df = df[cols]
    
    return df

In [1077]:
def create_full_region_column(df: pd.DataFrame, column: str = "행정구역별(시군구)", drop_original: bool = False) -> pd.DataFrame:
    """
    시도 + 시군구 조합으로 'SGG_NAME' 컬럼 생성 (항상 재생성)

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        column (str): 시도/시군구가 포함된 컬럼명
        drop_original (bool): 원본 column을 삭제할지 여부 (기본: False)

    Returns:
        pd.DataFrame: 'SGG_NAME' 컬럼이 포함된 DataFrame
    """
    cities = [
    "서울특별시", "인천광역시", "경기도"
    ]

    current_city = None
    new_names = []

    for idx, region in enumerate(df[column]):
        region = str(region).strip()
        if region in cities:
            current_city = region
            new_names.append(region)
        elif current_city is not None:
            new_names.append(f"{current_city} {region}")
        else:
            raise ValueError(f"[{idx}행] 시도 정보 없이 시군구 '{region}'가 나왔습니다.")


    # 항상 덮어쓰기
    df["SGG_NAME"] = new_names

    # 컬럼 정리
    if drop_original:
        df.drop(columns=[column], inplace=True)

    # 컬럼 순서 조정
    first_col = ["SGG_NAME"]
    if not drop_original and column != "SGG_NAME":
        first_col.append(column)
    remaining_cols = [col for col in df.columns if col not in first_col]
    df = df[first_col + remaining_cols]

    return df


In [1078]:
# 데이터 확인 및 저장
def validate_and_save_dataframe(
    df: pd.DataFrame,
    expected_rows: int = 66,
    region_col: str = "SGG_NAME",
    save_path: str = "./result.csv"
) -> None:
    """
    DataFrame의 행 개수가 기대값과 일치하면 정렬 후 CSV로 저장
    일치하지 않으면 경고 메시지만 출력

    Parameters:
        df (pd.DataFrame): 대상 데이터프레임
        expected_rows (int): 기대하는 행 개수 (기본값: 164)
        region_col (str): 정렬 기준 컬럼명 (기본값: 'SGG_NAME')
        save_path (str): 저장 경로 (기본값: './result.csv')
    """
    if len(df) == expected_rows:
        df = df.sort_values(by=region_col).reset_index(drop=True)
        df.to_csv(save_path, encoding="utf-8-sig", index=False)
        print(f"✅ 저장 완료: {save_path} (총 {expected_rows}개 행)")
    else:
        print(f"⚠️ 행 개수 불일치: 현재 {len(df)}개 행 (예상: {expected_rows})")


In [1079]:
# 시도와 시군구 컬럼을 결합하여 하나의 'SGG_NAME' 컬럼을 생성
def combine_region_columns(
    df: pd.DataFrame,
    col_sido: str = "시군구별(1)",
    col_sigungu: str = "시군구별(2)",
    new_col: str = "시군구별"
) -> pd.DataFrame:
    """
    시도와 시군구 컬럼을 결합하여 하나의 'SGG_NAME' 컬럼을 생성

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        col_sido (str): 시도 컬럼명 (기본값: '시군구별(1)')
        col_sigungu (str): 시군구 컬럼명 (기본값: '시군구별(2)')
        new_col (str): 생성될 컬럼명 (기본값: '시군구_전체')

    Returns:
        pd.DataFrame: 시군구_전체 컬럼이 추가된 DataFrame
    """
    # 0. 소계 제거 (단, 세종특별자치시 소계는 보존)
    df = df[~((df["시군구별(2)"] == "소계") & (df["시군구별(1)"] != "세종특별자치시"))]
    # df = df[~((df["시군구별(2)"] == "소계"))]
    
    # 1. 시군구 결합
    df[new_col] = df[col_sido].str.strip() + " " + df[col_sigungu].str.strip()

    # 2. 기존 컬럼 제거
    df.drop(columns=[col_sido, col_sigungu], inplace=True)

    # 3. 새 컬럼을 가장 왼쪽으로 이동
    cols = [new_col] + [col for col in df.columns if col != new_col]
    df = df[cols]

    return df


In [1080]:
# 시도 이름 정식 명칭으로 치환
def replace_abbreviated_sido_names(df: pd.DataFrame, column: str, col_cnt: int) -> pd.DataFrame:
    """
    주어진 컬럼에서 시도 약칭을 정식 명칭으로 치환

    Parameters:
        df (pd.DataFrame): 대상 DataFrame
        column (str): 변환할 컬럼명 ("시군구별(1)" 또는 "시군구별")

    Returns:
        pd.DataFrame: 치환된 DataFrame
    """
    mapping = {
        "서울": "서울특별시", "인천": "인천광역시", "경기": "경기도"
}

    if column not in df.columns:
        raise KeyError(f"'{column}' 컬럼없음")

    def convert_region_name(value):
        value = str(value).strip()
        for short, full in mapping.items():
            if value.startswith(full):
                return value  # 이미 정식 명칭이면 그대로 반환
            # 정확히 "강원도", "충북도"처럼 전체 단어 매칭
            if value.startswith(short + "도"):
                return value.replace(short + "도", full, 1)
            # 또는 "광역시" 등 약칭일 때 처리
            if value.startswith(short):
                return f"{full} {value[len(short):].strip()}"
        return value

    if col_cnt == 1:
        df[column] = df[column].apply(convert_region_name)
        df.rename(columns={column: "SGG_NAME"}, inplace=True)
    else:
        df[column] = df[column].apply(convert_region_name)
        df.rename(columns={"시군구별": "SGG_NAME"}, inplace=True)

    # 2. 중복된 '세종특별자치시'들 중 값 있는 쪽으로 채우기
    # df = df.groupby("SGG_NAME", as_index=False).first()

    # 정렬 수행
    # df = df.sort_values(by="SGG_NAME").reset_index(drop=True)
    
    # print(df)
    return df

In [1081]:
# 전처리 잘됬는지 확인하고 맞으면 저장
def validate_and_save_dataframe(
    df: pd.DataFrame,
    expected_rows: int = 66,
    region_col: str = "SGG_NAME",
    save_path: str = "./result.csv"
) -> None:
    """
    DataFrame의 행 개수가 기대값과 일치하면 정렬 후 CSV로 저장합니다.
    일치하지 않으면 경고 메시지만 출력합니다.

    Parameters:
        df (pd.DataFrame): 대상 데이터프레임
        expected_rows (int): 기대하는 행 개수 (기본값: 164)
        region_col (str): 정렬 기준 컬럼명 (기본값: 'SGG_NAME')
        save_path (str): 저장 경로 (기본값: './result.csv')
    """
    if len(df) == expected_rows:
        df = df.sort_values(by=region_col).reset_index(drop=True)
        # df.to_csv(save_path, encoding="utf-8-sig", index=False)
        # print(f"✅ 저장 완료: {save_path} (총 {expected_rows}개 행)")
    else:
        print(f"⚠️ 행 개수 불일치: 현재 {len(df)}개 행 (예상: {expected_rows})")

In [1082]:
def merge_subdistricts_to_city(df: pd.DataFrame, region_col: str = "SGG_NAME") -> pd.DataFrame:
    """
    구 단위 데이터를 시 단위로 병합하는 함수

    Parameters:
        df (pd.DataFrame): 원본 데이터프레임
        region_col (str): 시군구 전체 이름 컬럼명 (예: "시군구(전체)", "SGG_NAME" 등)

    Returns:
        pd.DataFrame: 병합 후 새로운 행 추가 및 기존 구 단위 행 제거된 DataFrame
    """
    merge_targets = {
    "경기도 수원시": [
        "경기도 수원시 장안구", "경기도 수원시 권선구", "경기도 수원시 팔달구", "경기도 수원시 영통구",
        "경기도 수원시장안구", "경기도 수원시권선구", "경기도 수원시팔달구", "경기도 수원시영통구",
        
    ],
    "경기도 고양시": [
        "경기도 고양시 덕양구", "경기도 고양시 일산동구", "경기도 고양시 일산서구",
        "경기도 고양시덕양구", "경기도 고양시일산동구", "경기도 고양시일산서구"
    ],
    "경기도 용인시": [
        "경기도 용인시 처인구", "경기도 용인시 기흥구", "경기도 용인시 수지구",
        "경기도 용인시처인구", "경기도 용인시기흥구", "경기도 용인시수지구"
    ],
    "경기도 성남시": [
        "경기도 성남시 수정구", "경기도 성남시 중원구", "경기도 성남시 분당구",
        "경기도 성남시수정구", "경기도 성남시중원구", "경기도 성남시분당구"
    ],
    "경기도 안산시": [
        "경기도 안산시 상록구", "경기도 안산시 단원구",
        "경기도 안산시상록구", "경기도 안산시단원구"
    ],
    "경기도 안양시": [
        "경기도 안양시 만안구", "경기도 안양시 동안구",
        "경기도 안양시만안구", "경기도 안양시동안구"
    ]
}

    for new_name, old_names in merge_targets.items():
        subset = df[df[region_col].isin(old_names)]
        if not subset.empty:
            summed = subset.drop(columns=[region_col]).sum(numeric_only=True)
            new_row = pd.DataFrame([{region_col: new_name, **summed.to_dict()}])
            df = pd.concat([df, new_row], ignore_index=True)
            print(f"병합 완료: {new_name}")

        # 기존 구 단위 삭제
        df = df[~df[region_col].isin(old_names)]
    
    # 정렬 수행
    df = df.sort_values(by=region_col).reset_index(drop=True)
    
    return df


In [1083]:
def check_missing_regions(df: pd.DataFrame, region_col: str = "SGG_NAME") -> pd.DataFrame:
    """
    지정된 시군구 리스트를 기준으로 CSV 파일 내 누락된 지역을 확인하고 저장 또는 출력하는 함수

    Parameters:
        csv_path (str): 확인할 CSV 파일 경로
        region_col (str): 지역명이 들어 있는 컬럼명 (예: 'SGG_NAME')
        region_list (list[str]): 기준이 되는 전체 시군구 리스트
        save_path (str): 누락된 항목을 저장할 경로 (지정하지 않으면 출력만 함)

    Returns:
        list[str]: 누락된 지역 목록
    """
    
    region_list = [
    "서울특별시 종로구", "서울특별시 중구", "서울특별시 용산구", "서울특별시 성동구",
    "서울특별시 광진구", "서울특별시 동대문구", "서울특별시 중랑구", "서울특별시 성북구",
    "서울특별시 강북구", "서울특별시 도봉구", "서울특별시 노원구", "서울특별시 은평구",
    "서울특별시 서대문구", "서울특별시 마포구", "서울특별시 양천구", "서울특별시 강서구",
    "서울특별시 구로구", "서울특별시 금천구", "서울특별시 영등포구", "서울특별시 동작구",
    "서울특별시 관악구", "서울특별시 서초구", "서울특별시 강남구", "서울특별시 송파구",
    "서울특별시 강동구",
    "경기도 가평군", "경기도 고양시", "경기도 과천시", "경기도 광명시", "경기도 광주시", 
    "경기도 구리시", "경기도 군포시", "경기도 김포시", "경기도 남양주시", "경기도 동두천시", 
    "경기도 부천시", "경기도 성남시", "경기도 수원시", "경기도 시흥시", "경기도 안산시",
    "경기도 안성시", "경기도 안양시", "경기도 양주시", "경기도 양평군", "경기도 여주시", 
    "경기도 연천군", "경기도 오산시", "경기도 용인시", "경기도 의왕시", "경기도 의정부시", 
    "경기도 이천시", "경기도 파주시", "경기도 평택시", "경기도 포천시", "경기도 하남시", 
    "경기도 화성시",
    "인천광역시 강화군", "인천광역시 옹진군", "인천광역시 계양구", "인천광역시 미추홀구",
    "인천광역시 남동구", "인천광역시 동구", "인천광역시 부평구", "인천광역시 서구",
    "인천광역시 연수구", "인천광역시 중구"
]

    df[region_col] = df[region_col].astype(str).str.strip()
    present_regions = set(df[region_col].unique())
    missing = sorted(set(region_list) - present_regions)

    if missing:
        # print(f"📋 누락된 시군구 {len(missing)}개 → 빈 행으로 추가됨:")
        # 누락된 시군구 이름만 있는 빈 DataFrame 생성
        missing_df = pd.DataFrame({region_col: missing})
        df = pd.concat([df, missing_df], ignore_index=True)
    # else:
    #    print("✅ 누락된 시군구 없음")
    # 정렬
    # 정렬 수행
    df = df.sort_values(by=region_col).reset_index(drop=True)
    return df


In [1084]:
def error_handling(df: pd.DataFrame) -> pd.DataFrame:

    remove_list = ["인천광역시", "서울특별시", "경기도", "nan",
                   "서울특별시 서울특별시", "경기도 여주군",
                   "경기도 경기도", "인천광역시 인천광역시"]
    df = df[~df["SGG_NAME"].isin(remove_list)]


    # 1. SGG_NAME이 NaN인 행 제거
    df = df.dropna(subset=["SGG_NAME"])
    
    df = df.replace("-", "")      # ''를 ""으로
    
    # 정렬 수행
    df = df.sort_values(by="SGG_NAME").reset_index(drop=True)
    
    return df


In [1085]:
# 첫 번째 컬럼명 확인 함수
def check_first_column(filepath: str):
    
    df = pd.read_csv(filepath, encoding="cp949") # 헤더 읽기
    first_col = df.columns[0] # 첫 번째 컬럼명
    
    if first_col == "시군구별(1)": # 법정동 컬럼 2개
        df = combine_region_columns(df) # 시군구 컬럼 2개 하나로 합치기(함수 제작 필요)
        # df.to_csv("./result1.csv", encoding="utf-8-sig", index=False)
        df = replace_abbreviated_sido_names(df, column="시군구별", col_cnt=2) # 시도 치환
        # df.to_csv("./result2.csv", encoding="utf-8-sig", index=False)
        df = merge_subdistricts_to_city(df, region_col="SGG_NAME") # 하위 행정구역 통합
        # df = create_full_region_column(df, column="시군구별") # 시군구_전체 컬럼 생성
        # df.to_csv("./result2.csv", encoding="utf-8-sig", index=False)
        #df.to_csv("./result2.csv", encoding="utf-8-sig", index=False)
        df = remove_target_regions(df, region_col="SGG_NAME") # 특례시 하위 행정구역 제거
        df = remove_only_sido_rows(df, region_col="SGG_NAME") # 시도만 있는 행 제거
    elif first_col =="시군구별": # 법정동 컬럼 1개
        df = replace_abbreviated_sido_names(df, column=first_col, col_cnt=1) # 시도 치환
        # df.to_csv("./result1.csv", encoding="utf-8-sig", index=False)
        # print(df)
        df = create_full_region_column(df, column="SGG_NAME") # 시군구_전체 컬럼 생성
        # df.to_csv("./result1.csv", encoding="utf-8-sig", index=False)
        df = remove_only_sido_rows(df, region_col="SGG_NAME") # 시도만 있는 행 제거 
        # df.to_csv("./result1.csv", encoding="utf-8-sig", index=False)
        # df.to_csv("./result1.csv", encoding="utf-8-sig", index=False)
        df = remove_target_regions(df, region_col="SGG_NAME") # 특례시 하위 행정구역 제거
        df = remove_only_sido_rows(df, region_col="SGG_NAME") # 시도만 있는 행 제거 

    else: # 둘다 아닐때 데이터 초기 전처리 잘못되었으므로 오류 발생 내용 추가
        pass # 향후 기능 추가
    
    df = check_missing_regions(df, region_col="SGG_NAME")
    df = error_handling(df)
    validate_and_save_dataframe(df)
    # df.to_csv("./result1.csv", encoding="utf-8-sig", index=False)
    df.to_csv("./result.csv", encoding="utf-8-sig", index=False)
    return df

In [1086]:
# 전처리 실행 코드(파일 하나)
# test1.csv -> 컬럼 2개, test.csv -> 컬럼 1개
# df = check_first_column('./data/test.csv') # 컬럼 2
# df = check_first_column('./data/education/교원_1인당_학생수_컬럼명변경.csv')

In [1087]:
def process_all_csvs_in_folder(folder_path: str, output_folder: str):
    """
    폴더 내 모든 .csv 파일을 읽어 전처리 후 output_folder에 저장

    Parameters:
        folder_path (str): .csv 파일이 있는 폴더 경로
        output_folder (str): 결과 저장할 폴더 경로
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            print(f"📄 처리 중: {filename}")
        
            # CSV 파일 읽기
            df = check_first_column(file_path)  # 사용자가 만든 전처리 함수

            # 결과 파일명 정의
            output_filename1 = filename.replace("_컬럼명변경", "")
            output_filename = output_filename1.replace(".csv", "_변경.csv")
            output_path = os.path.join(output_folder, output_filename)

            # 저장
            df.to_csv(output_path, index=False, encoding="utf-8-sig")
            print(f"✅ 저장 완료 → {output_path}")

#            except Exception as e:
#                print(f"❌ 오류 발생 ({filename}): {e}")


In [1088]:
# 전처리 실행 코드(파일 하나)
# test1.csv -> 컬럼 2개, test.csv -> 컬럼 1개
# df = check_first_column('./data/test.csv') # 컬럼 2
# df = check_first_column('./data/education/교원_1인당_학생수_컬럼명변경.csv')

In [1089]:
# 경제활동, 공공질서및안전 둘은 했으니까 실행XX

In [1090]:
input_path = "./sdg_test/data/감염병"
output_path = "./sdg_test/result/감염병"

process_all_csvs_in_folder(input_path, output_path)

📄 처리 중: 감염병 발생건수_수도권_정리.csv
⚠️ 행 개수 불일치: 현재 68개 행 (예상: 66)
✅ 저장 완료 → ./sdg_test/result/감염병\감염병 발생건수_수도권_정리_변경.csv


In [ ]:
region_list = [
    "강원특별자치도 강릉시", "강원특별자치도 고성군", "강원특별자치도 동해시", "강원특별자치도 삼척시", "강원특별자치도 속초시",
    "강원특별자치도 양구군", "강원특별자치도 양양군", "강원특별자치도 영월군", "강원특별자치도 원주시", "강원특별자치도 인제군",
    "강원특별자치도 정선군", "강원특별자치도 철원군", "강원특별자치도 춘천시", "강원특별자치도 태백시", "강원특별자치도 평창군",
    "강원특별자치도 홍천군", "강원특별자치도 화천군", "강원특별자치도 횡성군", "경상남도 거제시", "경상남도 거창군",
    "경상남도 고성군", "경상남도 김해시", "경상남도 남해군", "경상남도 밀양시", "경상남도 사천시", "경상남도 산청군",
    "경상남도 양산시", "경상남도 의령군", "경상남도 진주시", "경상남도 창녕군", "경상남도 창원시", "경상남도 통영시",
    "경상남도 하동군", "경상남도 함안군", "경상남도 함양군", "경상남도 합천군", "경상북도 경산시", "경상북도 경주시",
    "경상북도 고령군", "경상북도 구미시", "경상북도 김천시", "경상북도 문경시", "경상북도 봉화군", "경상북도 상주시",
    "경상북도 성주군", "경상북도 안동시", "경상북도 영덕군", "경상북도 영양군", "경상북도 영주시", "경상북도 영천시",
    "경상북도 예천군", "경상북도 울릉군", "경상북도 울진군", "경상북도 의성군", "경상북도 청도군", "경상북도 청송군",
    "경상북도 칠곡군", "경상북도 포항시", "광주광역시 광산구", "광주광역시 남구", "광주광역시 동구", "광주광역시 북구",
    "광주광역시 서구", "대구광역시 군위군", "대구광역시 남구", "대구광역시 달서구", "대구광역시 달성군", "대구광역시 동구",
    "대구광역시 북구", "대구광역시 서구", "대구광역시 수성구", "대구광역시 중구", "대전광역시 대덕구", "대전광역시 동구",
    "대전광역시 서구", "대전광역시 유성구", "대전광역시 중구", "부산광역시 강서구", "부산광역시 금정구", "부산광역시 기장군",
    "부산광역시 남구", "부산광역시 동구", "부산광역시 동래구", "부산광역시 북구", "부산광역시 사상구", "부산광역시 사하구",
    "부산광역시 서구", "부산광역시 수영구", "부산광역시 연제구", "부산광역시 영도구", "부산광역시 중구", "부산광역시 부산진구",
    "부산광역시 해운대구", "세종특별자치시", "울산광역시 남구", "울산광역시 동구", "울산광역시 북구", "울산광역시 울주군",
    "울산광역시 중구", "전라남도 강진군", "전라남도 고흥군", "전라남도 곡성군", "전라남도 광양시", "전라남도 구례군",
    "전라남도 나주시", "전라남도 담양군", "전라남도 목포시", "전라남도 무안군", "전라남도 보성군", "전라남도 순천시",
    "전라남도 신안군", "전라남도 여수시", "전라남도 영광군", "전라남도 영암군", "전라남도 완도군", "전라남도 장성군",
    "전라남도 장흥군", "전라남도 진도군", "전라남도 함평군", "전라남도 해남군", "전라남도 화순군", "전북특별자치도 고창군",
    "전북특별자치도 군산시", "전북특별자치도 김제시", "전북특별자치도 남원시", "전북특별자치도 무주군", "전북특별자치도 부안군",
    "전북특별자치도 순창군", "전북특별자치도 완주군", "전북특별자치도 익산시", "전북특별자치도 임실군", "전북특별자치도 장수군",
    "전북특별자치도 전주시", "전북특별자치도 정읍시", "전북특별자치도 진안군", "제주특별자치도 서귀포시", "제주특별자치도 제주시",
    "충청남도 계룡시", "충청남도 공주시", "충청남도 금산군", "충청남도 논산시", "충청남도 당진시", "충청남도 보령시",
    "충청남도 부여군", "충청남도 서산시", "충청남도 서천군", "충청남도 아산시", "충청남도 예산군", "충청남도 천안시",
    "충청남도 청양군", "충청남도 태안군", "충청남도 홍성군", "충청북도 괴산군", "충청북도 단양군", "충청북도 보은군",
    "충청북도 영동군", "충청북도 옥천군", "충청북도 음성군", "충청북도 제천시", "충청북도 증평군", "충청북도 진천군",
    "충청북도 청주시", "충청북도 충주시",
    "서울특별시 종로구", "서울특별시 중구", "서울특별시 용산구", "서울특별시 성동구",
    "서울특별시 광진구", "서울특별시 동대문구", "서울특별시 중랑구", "서울특별시 성북구",
    "서울특별시 강북구", "서울특별시 도봉구", "서울특별시 노원구", "서울특별시 은평구",
    "서울특별시 서대문구", "서울특별시 마포구", "서울특별시 양천구", "서울특별시 강서구",
    "서울특별시 구로구", "서울특별시 금천구", "서울특별시 영등포구", "서울특별시 동작구",
    "서울특별시 관악구", "서울특별시 서초구", "서울특별시 강남구", "서울특별시 송파구",
    "서울특별시 강동구",
    "경기도 가평군", "경기도 고양시", "경기도 과천시", "경기도 광명시", "경기도 광주시", 
    "경기도 구리시", "경기도 군포시", "경기도 김포시", "경기도 남양주시", "경기도 동두천시", 
    "경기도 부천시", "경기도 성남시", "경기도 수원시", "경기도 시흥시", "경기도 안산시",
    "경기도 안성시", "경기도 안양시", "경기도 양주시", "경기도 양평군", "경기도 여주시", 
    "경기도 연천군", "경기도 오산시", "경기도 용인시", "경기도 의왕시", "경기도 의정부시", 
    "경기도 이천시", "경기도 파주시", "경기도 평택시", "경기도 포천시", "경기도 하남시", 
    "경기도 화성시",
    "인천광역시 강화군", "인천광역시 옹진군", "인천광역시 계양구", "인천광역시 미추홀구",
    "인천광역시 남동구", "인천광역시 동구", "인천광역시 부평구", "인천광역시 서구",
    "인천광역시 연수구", "인천광역시 중구"
]